# 具有记忆功能的聊天机器人

## 回顾

[记忆](https://pmc.ncbi.nlm.nih.gov/articles/PMC10410470/)是一种认知功能，允许人们存储、检索和使用信息来理解他们的现在和未来。

有[各种长期记忆类型](https://langchain-ai.github.io/langgraph/concepts/memory/#memory)可以在AI应用中使用。

## 目标

在这里，我们将介绍[LangGraph Memory Store](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)作为保存和检索长期记忆的方法。

我们将构建一个使用`短期（线程内）`和`长期（跨线程）`记忆的聊天机器人。

我们将专注于长期[语义记忆](https://langchain-ai.github.io/langgraph/concepts/memory/#semantic-memory)，这将是关于用户的事实。

这些长期记忆将用于创建一个个性化的聊天机器人，可以记住关于用户的事实。

它将["在热路径中"](https://langchain-ai.github.io/langgraph/concepts/memory/#writing-memories)保存记忆，当用户与之聊天时。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph langchain_core

我们将使用[LangSmith](https://docs.smith.langchain.com/)进行[跟踪](https://docs.smith.langchain.com/concepts/tracing)。

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## LangGraph Store介绍

[LangGraph Memory Store](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)提供了一种在LangGraph中*跨线程*存储和检索信息的方法。

这是一个用于持久化`键值`存储的[开源基类](https://blog.langchain.dev/launching-long-term-memory-support-in-langgraph/)。

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore
in_memory_store = InMemoryStore()

在[Store](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)中存储对象（例如，记忆）时，我们提供：

- 对象的`namespace`，一个元组（类似于目录）
- 对象`key`（类似于文件名）
- 对象`value`（类似于文件内容）

我们使用[put](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore.put)方法通过`namespace`和`key`将对象保存到存储中。

![langgraph_store.png](attachment:6281b4e3-4930-467e-83ce-ba1aa837ca16.png)

In [ ]:
# 要保存的记忆的命名空间
user_id = "1"
namespace_for_memory = (user_id, "memories")

# 将记忆保存到命名空间作为键和值
key = str(uuid.uuid4())

# 值需要是一个字典
value = {"food_preference" : "我喜欢比萨饼"}

# 保存记忆
in_memory_store.put(namespace_for_memory, key, value)

我们使用[search](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore.search)通过`namespace`从存储中检索对象。

这返回一个列表。

In [ ]:
# 搜索
memories = in_memory_store.search(namespace_for_memory)
type(memories)

In [ ]:
# 元数据
memories[0].dict()

In [ ]:
# 键，值
print(memories[0].key, memories[0].value)

我们也可以使用[get](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore.get)通过`namespace`和`key`检索对象。

In [ ]:
# 通过命名空间和键获取记忆
memory = in_memory_store.get(namespace_for_memory, key)
memory.dict()

## 具有长期记忆的聊天机器人

我们想要一个[具有两种类型记忆](https://docs.google.com/presentation/d/181mvjlgsnxudQI6S3ritg9sooNyu4AcLLFH1UK0kIuk/edit#slide=id.g30eb3c8cf10_0_156)的聊天机器人：

1. `短期（线程内）记忆`：聊天机器人可以持久化对话历史和/或允许在聊天会话中中断。
2. `长期（跨线程）记忆`：聊天机器人可以在*所有聊天会话*中记住关于特定用户的信息。

In [ ]:
_set_env("OPENAI_API_KEY")

对于`短期记忆`，我们将使用[checkpointer](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpointer-libraries)。

参见模块2和我们的[概念文档](https://langchain-ai.github.io/langgraph/concepts/persistence/)了解更多关于checkpointers的信息，但总结如下：

* 它们在每个步骤将图状态写入线程。
* 它们在线程中持久化聊天历史。
* 它们允许图在线程中的任何步骤被中断和/或恢复。

并且，对于`长期记忆`，我们将使用如上所述的[LangGraph Store](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)。

In [ ]:
# 聊天模型
from langchain_openai import ChatOpenAI

# 初始化LLM
model = ChatOpenAI(model="gpt-4o", temperature=0)

聊天历史将使用checkpointer保存到短期记忆中。

聊天机器人将对聊天历史进行反思。

然后它将创建并保存记忆到[LangGraph Store](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.BaseStore)。

这个记忆在未来的聊天会话中可访问，以个性化聊天机器人的响应。

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.base import BaseStore

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables.config import RunnableConfig

# 聊天机器人指令
MODEL_SYSTEM_MESSAGE = """您是一个具有记忆功能的有用助手，提供有关用户的信息。
如果您有此用户的记忆，请使用它来个性化您的响应。
这是记忆（可能为空）：{memory}"""

# 从聊天历史和任何现有记忆创建新记忆
CREATE_MEMORY_INSTRUCTION = """您正在收集关于用户的信息以个性化您的响应。

当前用户信息：
{memory}

指令：
1. 仔细审查下面的聊天历史
2. 识别关于用户的新信息，例如：
   - 个人详细信息（姓名、位置）
   - 偏好（喜欢、不喜欢）
   - 兴趣和爱好
   - 过去的经历
   - 目标或未来计划
3. 将任何新信息与现有记忆合并
4. 将记忆格式化为清晰的项目符号列表
5. 如果新信息与现有记忆冲突，保留最新版本

记住：只包括用户直接陈述的事实信息。不要做假设或推断。

基于下面的聊天历史，请更新用户信息："""

def call_model(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """从存储中加载记忆并使用它来个性化聊天机器人的响应。"""
    
    # 从配置中获取用户ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索记忆
    namespace = ("memory", user_id)
    key = "user_memory"
    existing_memory = store.get(namespace, key)

    # 如果存在，提取实际记忆内容并添加前缀
    if existing_memory:
        # 值是一个带有记忆键的字典
        existing_memory_content = existing_memory.value.get('memory')
    else:
        existing_memory_content = "未找到现有记忆。"

    # 在系统提示中格式化记忆
    system_msg = MODEL_SYSTEM_MESSAGE.format(memory=existing_memory_content)
    
    # 使用记忆和聊天历史进行响应
    response = model.invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": response}

def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """反思聊天历史并将记忆保存到存储中。"""
    
    # 从配置中获取用户ID
    user_id = config["configurable"]["user_id"]

    # 从存储中检索现有记忆
    namespace = ("memory", user_id)
    existing_memory = store.get(namespace, "user_memory")
        
    # 提取记忆
    if existing_memory:
        existing_memory_content = existing_memory.value.get('memory')
    else:
        existing_memory_content = "未找到现有记忆。"

    # 在系统提示中格式化记忆
    system_msg = CREATE_MEMORY_INSTRUCTION.format(memory=existing_memory_content)
    new_memory = model.invoke([SystemMessage(content=system_msg)]+state['messages'])

    # 覆盖存储中的现有记忆
    key = "user_memory"

    # 将值写入为带有记忆键的字典
    store.put(namespace, key, {"memory": new_memory.content})

# 定义图
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_node("write_memory", write_memory)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", "write_memory")
builder.add_edge("write_memory", END)

# 长期（跨线程）记忆的存储
across_thread_memory = InMemoryStore()

# 短期（线程内）记忆的检查点
within_thread_memory = MemorySaver()

# 使用检查点和存储编译图
graph = builder.compile(checkpointer=within_thread_memory, store=across_thread_memory)

# 查看
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

当我们与聊天机器人交互时，我们提供两样东西：

1. `短期（线程内）记忆`：用于持久化聊天历史的`线程ID`。
2. `长期（跨线程）记忆`：用于将长期记忆命名空间到用户的`用户ID`。

让我们看看这些在实践中如何协同工作。

In [ ]:
# 我们为短期（线程内）记忆提供线程ID
# 我们为长期（跨线程）记忆提供用户ID
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="你好，我的名字是Lance")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="我喜欢在旧金山骑自行车")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

我们使用`MemorySaver`检查点进行线程内记忆。

这将聊天历史保存到线程中。

我们可以查看保存到线程的聊天历史。

In [ ]:
thread = {"configurable": {"thread_id": "1"}}
state = graph.get_state(thread).values
for m in state["messages"]: 
    m.pretty_print()

回想一下，我们使用存储编译了图：

```python
across_thread_memory = InMemoryStore()
```

并且，我们向图中添加了一个节点（`write_memory`），它反思聊天历史并将记忆保存到存储中。

我们可以看看记忆是否已保存到存储中。

In [ ]:
# 要保存的记忆的命名空间
user_id = "1"
namespace = ("memory", user_id)
existing_memory = across_thread_memory.get(namespace, "user_memory")
existing_memory.dict()

现在，让我们启动一个具有*相同用户ID*的*新线程*。

我们应该看到聊天机器人记住了用户的档案并使用它来个性化响应。

In [ ]:
# 我们为跨线程记忆提供用户ID以及新的线程ID
config = {"configurable": {"thread_id": "2", "user_id": "1"}}

# 用户输入
input_messages = [HumanMessage(content="你好！你推荐我去哪里骑自行车？")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

In [ ]:
# 用户输入
input_messages = [HumanMessage(content="太好了，附近有什么面包店我可以去看看吗？我喜欢骑车后吃羊角面包。")]

# 运行图
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

## 在LangSmith中查看跟踪

我们可以看到记忆从存储中检索并作为系统提示的一部分提供，如预期的那样：

https://smith.langchain.com/public/10268d64-82ff-434e-ac02-4afa5cc15432/r

## Studio

我们也可以在Studio中与我们的聊天机器人交互。

![Screenshot 2024-10-28 at 10.08.27 AM.png](attachment:afa216f7-4b67-4783-82af-c319e0f512ac.png)